### **Import Libraries**

Install required packages

In [1]:
!pip install -q xgboost shap lime joblib psutil

Import libraries

In [2]:
import os
import time
import json
import warnings
import joblib
import psutil
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


Load CSV

In [3]:
df = pd.read_csv('/content/Diabetes Dataset.csv')

print("Dataset loaded successfully.")

Dataset loaded successfully.


Inspect Dataset

In [4]:
df.head()

,Gender,Age,Physical Activity,Smoking Status,Alcohol Intake,Glucose,Blood Pressure,Skin Thickness,Insulin,BMI,Cholesterol,Diabetes Pedigree Function,Family History,Hypertension,Outcome
0,Male,63,Moderate,Former,NaN,103,40,7,41,28.196216,167,1.393,Yes,Yes,Diabetic
1,Female,51,Moderate,Never,Regular,104,59,21,51,30.438114,209,0.815,No,No,Diabetic
2,Female,74,Moderate,Current,NaN,128,70,18,199,38.700428,170,0.244,No,No,Diabetic
3,Male,78,Low,Never,Occasional,148,72,15,110,44.861600,170,0.336,No,No,Diabetic
4,Male,32,High,Never,NaN,102,55,18,49,21.591572,211,1.146,Yes,No,Non-diabetic


In [5]:
df.tail()

,Gender,Age,Physical Activity,Smoking Status,Alcohol Intake,Glucose,Blood Pressure,Skin Thickness,Insulin,BMI,Cholesterol,Diabetes Pedigree Function,Family History,Hypertension,Outcome
19995,Male,56,Moderate,Former,NaN,156,69,33,134,31.303926,257,0.112,No,No,Diabetic
19996,Male,67,Moderate,Current,NaN,117,79,15,67,38.402759,175,1.443,No,No,Diabetic
19997,Female,66,Moderate,Never,Regular,168,40,30,161,38.260875,193,2.238,No,No,Diabetic
19998,Female,68,Moderate,Never,NaN,130,63,27,105,35.598121,170,1.522,Yes,Yes,Diabetic
19999,Male,25,Low,Current,Regular,122,60,7,27,39.384417,200,0.303,No,No,Diabetic


In [6]:
df.shape

(20000, 15)

In [7]:
df.dtypes

,0
Gender,object
Age,int64
Physical Activity,object
Smoking Status,object
Alcohol Intake,object
Glucose,int64
Blood Pressure,int64
Skin Thickness,int64
Insulin,int64
BMI,float64


In [8]:
df.columns.tolist()

['Gender',
 'Age',
 'Physical Activity',
 'Smoking Status',
 'Alcohol Intake',
 'Glucose',
 'Blood Pressure',
 'Skin Thickness',
 'Insulin',
 'BMI',
 'Cholesterol',
 'Diabetes Pedigree Function',
 'Family History',
 'Hypertension',
 'Outcome']

In [9]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Gender,20000,2,Male,10028,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,20000.0,NaN,NaN,NaN,49.5654,17.320956,20.0,35.0,50.0,64.0,79.0
Physical Activity,20000,3,Moderate,9976,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Smoking Status,20000,3,Never,11999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Alcohol Intake,10075,2,Occasional,6014,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Glucose,20000.0,NaN,NaN,NaN,119.6034,29.590699,40.0,99.0,120.0,140.0,200.0
Blood Pressure,20000.0,NaN,NaN,NaN,69.43285,12.016115,30.0,61.0,69.0,78.0,114.0
Skin Thickness,20000.0,NaN,NaN,NaN,20.10545,9.130969,7.0,13.0,20.0,26.0,57.0
Insulin,20000.0,NaN,NaN,NaN,83.95255,53.026841,15.0,40.0,79.0,120.0,276.0
BMI,20000.0,NaN,NaN,NaN,32.00898,6.913271,15.0,27.314424,31.99404,36.750196,50.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Gender                      20000 non-null  object 
 1   Age                         20000 non-null  int64  
 2   Physical Activity           20000 non-null  object 
 3   Smoking Status              20000 non-null  object 
 4   Alcohol Intake              10075 non-null  object 
 5   Glucose                     20000 non-null  int64  
 6   Blood Pressure              20000 non-null  int64  
 7   Skin Thickness              20000 non-null  int64  
 8   Insulin                     20000 non-null  int64  
 9   BMI                         20000 non-null  float64
 10  Cholesterol                 20000 non-null  int64  
 11  Diabetes Pedigree Function  20000 non-null  float64
 12  Family History              20000 non-null  object 
 13  Hypertension                200

### **Data Quality Analysis**

Missing values

In [11]:
missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df)) * 100
})

display(missing_table)

,Missing Values,Missing Percentage
Gender,0,0.000
Age,0,0.000
Physical Activity,0,0.000
Smoking Status,0,0.000
Alcohol Intake,9925,49.625
Glucose,0,0.000
Blood Pressure,0,0.000
Skin Thickness,0,0.000
Insulin,0,0.000
BMI,0,0.000


Missing-Value Treatment

In [12]:
df["Alcohol Intake"] = df["Alcohol Intake"].fillna("Unknown")

missing_values_after = df.isnull().sum()

display(
    pd.DataFrame({
        "Missing Values": missing_values_after,
        "Missing Percentage": (missing_values_after / len(df)) * 100
    })
)

,Missing Values,Missing Percentage
Gender,0,0.0
Age,0,0.0
Physical Activity,0,0.0
Smoking Status,0,0.0
Alcohol Intake,0,0.0
Glucose,0,0.0
Blood Pressure,0,0.0
Skin Thickness,0,0.0
Insulin,0,0.0
BMI,0,0.0


Duplicate records

In [73]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)
print("Shape after removing duplicates:", df.shape)

Number of duplicate rows: 0
Shape after removing duplicates: (20000, 15)


Unique Outcome values

In [15]:
print("Unique Outcome values:")
print(df["Outcome"].value_counts(dropna=False))

Unique Outcome values:
Outcome
Diabetic        16726
Non-diabetic     3274
Name: count, dtype: int64


Clean target

In [16]:
df["Outcome"] = df["Outcome"].astype(str).str.strip()

target_mapping = {
    "Diabetic": 1,
    "Non-diabetic": 0
}

df["Outcome"] = df["Outcome"].map(target_mapping)

print(df["Outcome"].value_counts(dropna=False))

Outcome
1    16726
0     3274
Name: count, dtype: int64


Remove invalid target records

In [17]:
df = df.dropna(subset=["Outcome"]).reset_index(drop=True)

df["Outcome"] = df["Outcome"].astype(int)

print("Final dataset shape:", df.shape)

Final dataset shape: (20000, 15)


### **Exploratory Data Analysis**

Target distribution

In [18]:
import plotly.express as px

outcome_counts = df["Outcome"].value_counts().reset_index()
outcome_counts.columns = ["Outcome", "Count"]

outcome_counts["Outcome Label"] = outcome_counts["Outcome"].map({
    0: "Non-diabetic",
    1: "Diabetic"
})

fig = px.bar(
    outcome_counts,
    x="Outcome Label",
    y="Count",
    color="Outcome Label",
    color_discrete_map={
        "Non-diabetic": "#3498DB",
        "Diabetic": "#E74C3C"
    },
    text="Count",
    title="Diabetes Outcome Distribution"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Outcome",
    yaxis_title="Number of Records",
    showlegend=False,
    template="plotly_white"
)

fig.show()

Numerical feature distributions

In [19]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

if "Outcome" in numerical_columns:
    numerical_columns.remove("Outcome")

colors = [
    "#3498DB",
    "#E74C3C",
    "#2ECC71",
    "#9B59B6",
    "#F39C12",
    "#1ABC9C",
    "#E67E22",
    "#34495E"
]

cols = 3
rows = int(np.ceil(len(numerical_columns) / cols))

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=numerical_columns
)

for i, column in enumerate(numerical_columns):
    row = (i // cols) + 1
    col = (i % cols) + 1

    fig.add_trace(
        go.Histogram(
            x=df[column],
            nbinsx=20,
            marker_color=colors[i % len(colors)],
            name=column
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Distribution of Numerical Features",
    height=350 * rows,
    width=1100,
    showlegend=False,
    template="plotly_white"
)

fig.update_xaxes(title_text="Value")
fig.update_yaxes(title_text="Frequency")

fig.show()

Categorical features

In [20]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Categorical features:")
print(categorical_columns)

Categorical features:
['Gender', 'Physical Activity', 'Smoking Status', 'Alcohol Intake', 'Family History', 'Hypertension']


Correlation matrix

In [21]:
import numpy as np
import plotly.graph_objects as go

correlation_matrix = df.select_dtypes(include=np.number).corr()

fig = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        text=np.round(correlation_matrix.values, 2),
        texttemplate="%{text}",
        textfont={"size": 11},
        hovertemplate=(
            "<b>%{y}</b> vs <b>%{x}</b><br>"
            "Correlation: %{z:.2f}"
            "<extra></extra>"
        ),
        colorbar=dict(
            title="Correlation"
        )
    )
)

fig.update_layout(
    title="Correlation Matrix",
    width=1000,
    height=800,
    template="plotly_white",
    xaxis=dict(
        title="Features",
        tickangle=-45
    ),
    yaxis=dict(
        title="Features"
    )
)

fig.show()

### **Target-Class Analysis**

In [22]:
class_distribution = df["Outcome"].value_counts()

class_percentage = df["Outcome"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Count": class_distribution,
    "Percentage": class_percentage.round(2)
})

target_summary.index = ["Non-diabetic", "Diabetic"]

display(target_summary)

,Count,Percentage
Non-diabetic,16726,83.63
Diabetic,3274,16.37


In [23]:
imbalance_ratio = class_distribution.max() / class_distribution.min()

print("Class imbalance ratio:", round(imbalance_ratio, 3))

Class imbalance ratio: 5.109


### **Encode Categorical Variables**

In [24]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['Gender', 'Physical Activity', 'Smoking Status', 'Alcohol Intake', 'Family History', 'Hypertension']

Numerical features:
['Age', 'Glucose', 'Blood Pressure', 'Skin Thickness', 'Insulin', 'BMI', 'Cholesterol', 'Diabetes Pedigree Function']


### **Separate Features and Target**

In [25]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget values:")
print(y.value_counts())

Feature matrix shape: (20000, 14)
Target shape: (20000,)

Target values:
Outcome
1    16726
0     3274
Name: count, dtype: int64


### **Train/Test Split**

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (16000, 14)
Testing set: (4000, 14)

Training target distribution:
Outcome
1    0.836313
0    0.163688
Name: proportion, dtype: float64

Testing target distribution:
Outcome
1    0.83625
0    0.16375
Name: proportion, dtype: float64


### **Preprocessing Pipeline**

In [27]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


Fit transformation

In [28]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (16000, 23)
Processed testing shape: (4000, 23)


Feature names

In [29]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 23
['numeric__Age' 'numeric__Glucose' 'numeric__Blood Pressure'
 'numeric__Skin Thickness' 'numeric__Insulin' 'numeric__BMI'
 'numeric__Cholesterol' 'numeric__Diabetes Pedigree Function'
 'categorical__Gender_Female' 'categorical__Gender_Male'
 'categorical__Physical Activity_High'
 'categorical__Physical Activity_Low'
 'categorical__Physical Activity_Moderate'
 'categorical__Smoking Status_Current'
 'categorical__Smoking Status_Former' 'categorical__Smoking Status_Never'
 'categorical__Alcohol Intake_Occasional'
 'categorical__Alcohol Intake_Regular'
 'categorical__Alcohol Intake_Unknown' 'categorical__Family History_No'
 'categorical__Family History_Yes' 'categorical__Hypertension_No'
 'categorical__Hypertension_Yes']


### **Logistic Regression**

In [30]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


### **Random Forest**

In [31]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


### **XGBoost**

In [32]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

print("XGBoost trained successfully.")

XGBoost trained successfully.


### **5-Fold Cross-Validation**

In [33]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgb_model
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = {}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_processed,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    cv_results[name] = {
        "Mean F1": scores.mean(),
        "Std F1": scores.std()
    }

    print(f"\n{name}")
    print("Fold F1 scores:", np.round(scores, 4))
    print("Mean F1:", round(scores.mean(), 4))
    print("Std F1:", round(scores.std(), 4))


Logistic Regression
Fold F1 scores: [0.986  0.986  0.9849 0.9893 0.9884]
Mean F1: 0.9869
Std F1: 0.0017

Random Forest
Fold F1 scores: [0.9823 0.984  0.9818 0.9866 0.9835]
Mean F1: 0.9836
Std F1: 0.0017

XGBoost
Fold F1 scores: [0.9864 0.9852 0.9845 0.9875 0.9871]
Mean F1: 0.9861
Std F1: 0.0011


In [34]:
cv_results_df = pd.DataFrame(cv_results).T

display(cv_results_df.round(4))

,Mean F1,Std F1
Logistic Regression,0.9869,0.0017
Random Forest,0.9836,0.0017
XGBoost,0.9861,0.0011


### **Hyperparameter Tuning**

Logistic Regression tuning

In [35]:
lr_grid = {
    "C": [0.01, 0.1, 1, 10]
}

lr_search = GridSearchCV(
    LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    lr_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

lr_search.fit(X_train_processed, y_train)

print("Best Logistic Regression parameters:")
print(lr_search.best_params_)

print("Best CV F1:",
      round(lr_search.best_score_, 4))

Best Logistic Regression parameters:
{'C': 0.1}
Best CV F1: 0.9872


Random Forest tuning

In [36]:
rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

rf_search = GridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    rf_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

rf_search.fit(X_train_processed, y_train)

print("Best Random Forest parameters:")
print(rf_search.best_params_)

print("Best CV F1:",
      round(rf_search.best_score_, 4))

Best Random Forest parameters:
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1: 0.9838


XGBoost tuning

In [37]:
xgb_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1]
}

xgb_search = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ),
    xgb_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

xgb_search.fit(X_train_processed, y_train)

print("Best XGBoost parameters:")
print(xgb_search.best_params_)

print("Best CV F1:",
      round(xgb_search.best_score_, 4))

Best XGBoost parameters:
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}
Best CV F1: 0.9866


Store tuned models

In [38]:
best_models = {
    "Logistic Regression": lr_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_
}

print("Tuned models stored successfully.")

Tuned models stored successfully.


### **Final Test-Set Evaluation**

In [39]:
results = {}

for name, model in best_models.items():

    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["Non-diabetic", "Diabetic"],
        zero_division=0
    ))


Logistic Regression
              precision    recall  f1-score   support

Non-diabetic       0.96      0.90      0.93       655
    Diabetic       0.98      0.99      0.99      3345

    accuracy                           0.98      4000
   macro avg       0.97      0.95      0.96      4000
weighted avg       0.98      0.98      0.98      4000


Random Forest
              precision    recall  f1-score   support

Non-diabetic       0.91      0.92      0.91       655
    Diabetic       0.98      0.98      0.98      3345

    accuracy                           0.97      4000
   macro avg       0.95      0.95      0.95      4000
weighted avg       0.97      0.97      0.97      4000


XGBoost
              precision    recall  f1-score   support

Non-diabetic       0.93      0.92      0.93       655
    Diabetic       0.99      0.99      0.99      3345

    accuracy                           0.98      4000
   macro avg       0.96      0.96      0.96      4000
weighted avg       0.98      

### **Model Comparison and Best Model**

In [40]:
results_df = pd.DataFrame(results).T

display(results_df.round(4))

,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Logistic Regression,0.9780,0.9811,0.9928,0.9869,0.9972
Random Forest,0.9715,0.9838,0.9821,0.9829,0.9954
XGBoost,0.9768,0.9851,0.9871,0.9861,0.9968


In [41]:
best_model_name = results_df["F1-Score"].idxmax()

best_model = best_models[best_model_name]

print("Best model:", best_model_name)

print("\nPerformance:")
display(
    results_df.loc[[best_model_name]].round(4)
)

Best model: Logistic Regression

Performance:


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Logistic Regression,0.978,0.9811,0.9928,0.9869,0.9972


Confusion matrices

In [42]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=list(best_models.keys())
)

colorscales = [
    "Blues",
    "Greens",
    "Purples"
]

for i, ((name, model), colorscale) in enumerate(
    zip(best_models.items(), colorscales),
    start=1
):

    predictions = model.predict(X_test_processed)

    cm = confusion_matrix(
        y_test,
        predictions
    )

    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=["Non-diabetic", "Diabetic"],
            y=["Non-diabetic", "Diabetic"],
            text=cm,
            texttemplate="%{text}",
            textfont={"size": 18},
            colorscale=colorscale,
            showscale=False,
            hovertemplate=(
                "<b>Actual:</b> %{y}<br>"
                "<b>Predicted:</b> %{x}<br>"
                "<b>Count:</b> %{z}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=i
    )

fig.update_layout(
    title="Confusion Matrix Comparison of Best Models",
    width=1400,
    height=500,
    template="plotly_white"
)

fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="Actual")

fig.show()

ROC curves

In [43]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, roc_auc_score

fig = go.Figure()

for name, model in best_models.items():


    probabilities = model.predict_proba(
        X_test_processed
    )[:, 1]


    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )


    auc = roc_auc_score(
        y_test,
        probabilities
    )


    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC = {auc:.3f})",
            hovertemplate=(
                "False Positive Rate: %{x:.3f}<br>"
                "True Positive Rate: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )


fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random Classifier",
        line=dict(
            dash="dash"
        ),
        hoverinfo="skip"
    )
)


fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=900,
    height=650,
    template="plotly_white",
    legend=dict(
        x=0.60,
        y=0.05
    )
)

fig.show()

### **SHAP Global + Local Analysis**

In [44]:
print("Selected model for XAI:", best_model_name)

Selected model for XAI: Logistic Regression


SHAP explainer

In [45]:
if best_model_name == "Logistic Regression":

    shap_explainer = shap.LinearExplainer(
        best_model,
        X_train_processed
    )

else:

    shap_explainer = shap.TreeExplainer(
        best_model
    )

print("SHAP explainer created.")

SHAP explainer created.


Calculate SHAP values

In [46]:
X_shap_sample = X_test_processed[
    :min(500, len(X_test_processed))
]

shap_values_raw = shap_explainer(
    X_shap_sample
)

print("SHAP values calculated.")

SHAP values calculated.


Handle binary-class SHAP output

In [47]:
if len(shap_values_raw.shape) == 3:
    shap_values_matrix = shap_values_raw[:, :, 1]
else:
    shap_values_matrix = shap_values_raw.values

print("SHAP matrix shape:", shap_values_matrix.shape)

SHAP matrix shape: (500, 23)


SHAP global importance

In [48]:
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": np.abs(
        shap_values_matrix
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

display(shap_importance.head(15))

,Feature,Mean_Absolute_SHAP
1,numeric__Glucose,6.288683
5,numeric__BMI,2.193117
0,numeric__Age,0.084942
6,numeric__Cholesterol,0.045023
7,numeric__Diabetes Pedigree Function,0.028718
11,categorical__Physical Activity_Low,0.027800
18,categorical__Alcohol Intake_Unknown,0.016917
22,categorical__Hypertension_Yes,0.016591
14,categorical__Smoking Status_Former,0.016090
9,categorical__Gender_Male,0.016051


Global SHAP plot

In [49]:
import plotly.express as px


top_features = shap_importance.head(15).copy()


fig = px.bar(
    top_features,
    x="Mean_Absolute_SHAP",
    y="Feature",
    orientation="h",
    color="Mean_Absolute_SHAP",
    color_continuous_scale="Blues",
    text="Mean_Absolute_SHAP",
    title="Top Global SHAP Features"
)


fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)


fig.update_layout(
    xaxis_title="Mean Absolute SHAP Value",
    yaxis_title="Feature",
    template="plotly_white",
    height=700,
    width=1000,
    coloraxis_colorbar_title="SHAP Value"
)


fig.update_yaxes(
    categoryorder="total ascending"
)

fig.show()

Local SHAP explanation

In [50]:
sample_index = 0

sample_shap_values = shap_values_matrix[sample_index]

local_shap = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Value": sample_shap_values
})

local_shap["Absolute_SHAP"] = np.abs(
    local_shap["SHAP_Value"]
)

local_shap = local_shap.sort_values(
    "Absolute_SHAP",
    ascending=False
)

display(local_shap.head(15))

,Feature,SHAP_Value,Absolute_SHAP
1,numeric__Glucose,2.344170,2.344170
5,numeric__BMI,-1.968273,1.968273
6,numeric__Cholesterol,-0.100736,0.100736
0,numeric__Age,-0.062520,0.062520
11,categorical__Physical Activity_Low,-0.050323,0.050323
7,numeric__Diabetes Pedigree Function,-0.046180,0.046180
12,categorical__Physical Activity_Moderate,-0.017342,0.017342
18,categorical__Alcohol Intake_Unknown,-0.016917,0.016917
9,categorical__Gender_Male,-0.016695,0.016695
8,categorical__Gender_Female,-0.014639,0.014639


### **LIME Local Analysis**

Create LIME explainer

In [51]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train_processed,
    feature_names=feature_names,
    class_names=["Non-diabetic", "Diabetic"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

print("LIME explainer created.")

LIME explainer created.


Explain one patient

In [52]:
lime_start = time.perf_counter()

lime_explanation = lime_explainer.explain_instance(
    X_test_processed[sample_index],
    best_model.predict_proba,
    num_features=15
)

lime_time = time.perf_counter() - lime_start

print("LIME explanation generated.")
print("LIME time:", round(lime_time, 4), "seconds")

LIME explanation generated.
LIME time: 0.0724 seconds


Display LIME explanation

In [53]:
lime_results = pd.DataFrame(
    lime_explanation.as_list(),
    columns=["Feature", "LIME_Weight"]
)

lime_results["Absolute_LIME"] = np.abs(
    lime_results["LIME_Weight"]
)

lime_results = lime_results.sort_values(
    "Absolute_LIME",
    ascending=False
)

display(lime_results)

,Feature,LIME_Weight,Absolute_LIME
0,0.01 < numeric__Glucose <= 0.69,0.250798,0.250798
1,numeric__BMI <= -0.67,-0.163769,0.163769
2,categorical__Alcohol Intake_Occasional <= 0.00,0.028453,0.028453
3,0.00 < categorical__Hypertension_No <= 1.00,-0.019826,0.019826
4,categorical__Alcohol Intake_Unknown <= 0.00,0.015793,0.015793
5,categorical__Physical Activity_High <= 0.00,0.009086,0.009086
6,categorical__Family History_Yes <= 0.00,-0.008026,0.008026
7,-0.84 < numeric__Age <= -0.03,0.006776,0.006776
8,categorical__Gender_Male <= 0.00,-0.005306,0.005306
9,-0.03 < numeric__Blood Pressure <= 0.71,-0.005246,0.005246


### **SHAP vs LIME Comparison**

Feature agreement

In [74]:
import re

top_n = 10


def extract_feature_name(feature):
    """
    Converts SHAP/LIME feature descriptions into
    comparable original feature names.
    """

    feature = str(feature)

    feature = re.sub(r'^.*<\s*(numeric__|categorical__)',
                     r'\1', feature)

    feature = re.sub(r'\s*(<=|>=|<|>)\s*-?\d+\.?\d*', '', feature)

    feature = feature.split(' <= ')[0]
    feature = feature.split(' > ')[0]

    feature = feature.replace('numeric__', '')
    feature = feature.replace('categorical__', '')

    categorical_features = [
        'Physical Activity',
        'Smoking Status',
        'Gender',
        'Alcohol Intake',
        'Family History',
        'Hypertension'
    ]

    for cat in categorical_features:
        if feature.startswith(cat + '_'):
            feature = cat

    return feature.strip()


shap_top_features_raw = (
    shap_importance
    .head(top_n)["Feature"]
    .tolist()
)

shap_top_features = set(
    extract_feature_name(f)
    for f in shap_top_features_raw
)

lime_top_features_raw = (
    lime_results
    .head(top_n)["Feature"]
    .tolist()
)

lime_top_features = set(
    extract_feature_name(f)
    for f in lime_top_features_raw
)


common_features = (
    shap_top_features &
    lime_top_features
)


feature_agreement = (
    len(common_features) /
    top_n
)


print("=" * 60)
print("SHAP vs LIME FEATURE AGREEMENT")
print("=" * 60)

print("\nSHAP top features:")
for f in sorted(shap_top_features):
    print("-", f)

print("\nLIME top features:")
for f in sorted(lime_top_features):
    print("-", f)

print("\nCommon features:")
for f in sorted(common_features):
    print("-", f)

print("\nNumber of common features:",
      len(common_features))

print("Top-N features:",
      top_n)

print("Feature Agreement:",
      round(feature_agreement, 4))

print("Feature Agreement (%):",
      round(feature_agreement * 100, 2), "%")

SHAP vs LIME FEATURE AGREEMENT

SHAP top features:
- Age
- Alcohol Intake
- BMI
- Cholesterol
- Diabetes Pedigree Function
- Gender
- Glucose
- Hypertension
- Physical Activity
- Smoking Status

LIME top features:
- Age
- Alcohol Intake
- BMI
- Blood Pressure
- Family History
- Gender
- Glucose
- Hypertension
- Physical Activity

Common features:
- Age
- Alcohol Intake
- BMI
- Gender
- Glucose
- Hypertension
- Physical Activity

Number of common features: 7
Top-N features: 10
Feature Agreement: 0.7
Feature Agreement (%): 70.0 %


SHAP consistency

In [55]:
consistency_sample_size = min(
    100,
    len(X_test_processed)
)

X_consistency = X_test_processed[
    :consistency_sample_size
]

shap_consistency_values = shap_explainer(
    X_consistency
)

if len(shap_consistency_values.shape) == 3:
    shap_consistency_matrix = (
        shap_consistency_values[:, :, 1]
    )
else:
    shap_consistency_matrix = (
        shap_consistency_values.values
    )

shap_rank_matrix = np.argsort(
    -np.abs(shap_consistency_matrix),
    axis=1
)

shap_top_feature_sets = []

for row in shap_rank_matrix:

    top_indices = row[:10]

    shap_top_feature_sets.append(
        set(feature_names[top_indices])
    )

print("SHAP consistency analysis completed.")

SHAP consistency analysis completed.


In [56]:
shap_jaccard_scores = []

for i in range(
    len(shap_top_feature_sets) - 1
):

    A = shap_top_feature_sets[i]
    B = shap_top_feature_sets[i + 1]

    union = A | B

    if len(union) == 0:
        score = 1
    else:
        score = len(A & B) / len(union)

    shap_jaccard_scores.append(score)

shap_consistency_score = np.mean(
    shap_jaccard_scores
)

print(
    "SHAP consistency score:",
    round(shap_consistency_score, 4)
)

SHAP consistency score: 0.5864


LIME consistency

In [57]:
lime_consistency_scores = []

lime_sample_size = min(
    30,
    len(X_test_processed)
)

for i in range(lime_sample_size):

    explanation = lime_explainer.explain_instance(
        X_test_processed[i],
        best_model.predict_proba,
        num_features=10
    )

    feature_set = set(
        [
            item[0]
            for item in explanation.as_list()
        ]
    )

    lime_consistency_scores.append(
        feature_set
    )

lime_jaccard = []

for i in range(
    len(lime_consistency_scores) - 1
):

    A = lime_consistency_scores[i]
    B = lime_consistency_scores[i + 1]

    union = A | B

    if len(union) == 0:
        score = 1
    else:
        score = len(A & B) / len(union)

    lime_jaccard.append(score)

lime_consistency_score = np.mean(
    lime_jaccard
)

print(
    "LIME consistency score:",
    round(lime_consistency_score, 4)
)

LIME consistency score: 0.1099


LIME fidelity

In [58]:
lime_fidelity_samples = min(
    30,
    len(X_test_processed)
)

lime_fidelity_scores = []

for i in range(lime_fidelity_samples):

    explanation = lime_explainer.explain_instance(
        X_test_processed[i],
        best_model.predict_proba,
        num_features=10
    )

    weights = dict(
        explanation.as_list()
    )

    baseline_probability = best_model.predict_proba(
        X_test_processed[i].reshape(1, -1)
    )[0, 1]

    surrogate_score = (
        explanation.score
        if hasattr(explanation, "score")
        else np.nan
    )

    if not np.isnan(surrogate_score):
        lime_fidelity_scores.append(
            surrogate_score
        )

lime_fidelity = (
    np.mean(lime_fidelity_scores)
    if lime_fidelity_scores
    else np.nan
)

print(
    "LIME fidelity:",
    round(lime_fidelity, 4)
    if not np.isnan(lime_fidelity)
    else "Not available"
)

LIME fidelity: 0.288


SHAP fidelity

In [59]:
X_fidelity = X_test_processed[
    :min(100, len(X_test_processed))
]

shap_fidelity_values = shap_explainer(
    X_fidelity
)

if len(shap_fidelity_values.shape) == 3:
    shap_fidelity_matrix = (
        shap_fidelity_values[:, :, 1]
    )
    base_values = np.array(
        shap_fidelity_values.base_values
    )

    if base_values.ndim > 1:
        base_values = base_values[:, 1]

else:
    shap_fidelity_matrix = (
        shap_fidelity_values.values
    )

    base_values = np.array(
        shap_fidelity_values.base_values
    )

model_probability = best_model.predict_proba(
    X_fidelity
)[:, 1]

shap_reconstructed = (
    base_values +
    shap_fidelity_matrix.sum(axis=1)
)

if len(shap_reconstructed) == len(model_probability):

    shap_fidelity = np.corrcoef(
        shap_reconstructed,
        model_probability
    )[0, 1]

else:

    shap_fidelity = np.nan

print(
    "SHAP fidelity:",
    round(shap_fidelity, 4)
    if not np.isnan(shap_fidelity)
    else "Not available"
)

SHAP fidelity: 0.7298


Computational Efficiency

In [60]:
start_time = time.perf_counter()

_ = shap_explainer(
    X_test_processed[
        :min(100, len(X_test_processed))
    ]
)

shap_runtime = time.perf_counter() - start_time



start_time = time.perf_counter()

for i in range(
    min(10, len(X_test_processed))
):

    _ = lime_explainer.explain_instance(
        X_test_processed[i],
        best_model.predict_proba,
        num_features=10
    )

lime_runtime = time.perf_counter() - start_time


print("SHAP runtime:", round(shap_runtime, 4), "seconds")
print("LIME runtime:", round(lime_runtime, 4), "seconds")

SHAP runtime: 0.0065 seconds
LIME runtime: 0.9544 seconds


### **Final SHAP vs LIME Comparison**

In [61]:
comparison_data = {
    "Criterion": [
        "Feature Agreement",
        "Consistency",
        "Fidelity",
        "Computational Efficiency"
    ],

    "SHAP": [
        feature_agreement,
        shap_consistency_score,
        shap_fidelity,
        shap_runtime
    ],

    "LIME": [
        feature_agreement,
        lime_consistency_score,
        lime_fidelity,
        lime_runtime
    ]
}

xai_comparison_df = pd.DataFrame(
    comparison_data
)

display(
    xai_comparison_df.round(4)
)

,Criterion,SHAP,LIME
0,Feature Agreement,0.7000,0.7000
1,Consistency,0.5864,0.1099
2,Fidelity,0.7298,0.2880
3,Computational Efficiency,0.0065,0.9544


Visualise SHAP vs LIME

In [62]:
import plotly.express as px

plot_df = xai_comparison_df.copy()

plot_df = plot_df[
    plot_df["Criterion"] != "Computational Efficiency"
]

plot_long = plot_df.melt(
    id_vars="Criterion",
    value_vars=["SHAP", "LIME"],
    var_name="Method",
    value_name="Score"
)

fig = px.bar(
    plot_long,
    x="Criterion",
    y="Score",
    color="Method",
    barmode="group",
    text="Score",
    title="SHAP vs LIME Comparison"
)
fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Evaluation Criterion",
    yaxis_title="Score",
    template="plotly_white",
    width=1000,
    height=600,
    legend_title="Explainability Method"
)

fig.show()

### **Technical CDSS Prototype**

Prediction function

In [63]:
def predict_diabetes_risk(patient_data):

    patient_df = pd.DataFrame(
        [patient_data]
    )

    processed_patient = preprocessor.transform(
        patient_df
    )

    probability = best_model.predict_proba(
        processed_patient
    )[0, 1]

    prediction = int(
        probability >= 0.50
    )

    if probability < 0.30:
        risk_level = "Low Risk"

    elif probability < 0.70:
        risk_level = "Moderate Risk"

    else:
        risk_level = "High Risk"

    return (
        prediction,
        probability,
        risk_level,
        processed_patient
    )

Sample patient

In [64]:
sample_patient = {
    "Gender": "Female",
    "Age": 51,
    "Physical Activity": "Moderate",
    "Smoking Status": "Never",
    "Alcohol Intake": "Regular",
    "Glucose": 104,
    "Blood Pressure": 59,
    "Skin Thickness": 21,
    "Insulin": 51,
    "BMI": 30.438,
    "Cholesterol": 209,
    "Diabetes Pedigree Function": 0.815,
    "Family History": "No",
    "Hypertension": "No"
}

prediction, probability, risk_level, processed_patient = (
    predict_diabetes_risk(sample_patient)
)

print("========== CDSS OUTPUT ==========")

print(
    "Prediction:",
    "Diabetic" if prediction == 1
    else "Non-diabetic"
)

print(
    "Predicted probability:",
    f"{probability:.2%}"
)

print(
    "Risk category:",
    risk_level
)

========== CDSS OUTPUT ==========
Prediction: Diabetic
Predicted probability: 96.91%
Risk category: High Risk


SHAP explanation for CDSS

In [65]:
patient_shap = shap_explainer(
    processed_patient
)

if len(patient_shap.shape) == 3:
    patient_shap_values = patient_shap[:, :, 1][0]
else:
    patient_shap_values = patient_shap.values[0]

patient_shap_df = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Value": patient_shap_values
})

patient_shap_df["Absolute_SHAP"] = np.abs(
    patient_shap_df["SHAP_Value"]
)

patient_shap_df = patient_shap_df.sort_values(
    "Absolute_SHAP",
    ascending=False
)

display(
    patient_shap_df.head(10)
)

,Feature,SHAP_Value,Absolute_SHAP
1,numeric__Glucose,-4.091777,4.091777
5,numeric__BMI,-0.644119,0.644119
6,numeric__Cholesterol,0.027947,0.027947
7,numeric__Diabetes Pedigree Function,0.023729,0.023729
11,categorical__Physical Activity_Low,0.017681,0.017681
18,categorical__Alcohol Intake_Unknown,-0.016917,0.016917
9,categorical__Gender_Male,-0.016695,0.016695
0,numeric__Age,0.015352,0.015352
12,categorical__Physical Activity_Moderate,0.014773,0.014773
8,categorical__Gender_Female,-0.014639,0.014639


LIME explanation for CDSS

In [66]:
patient_lime = lime_explainer.explain_instance(
    processed_patient[0],
    best_model.predict_proba,
    num_features=10
)

patient_lime_df = pd.DataFrame(
    patient_lime.as_list(),
    columns=[
        "Feature",
        "LIME_Weight"
    ]
)

patient_lime_df["Absolute_LIME"] = np.abs(
    patient_lime_df["LIME_Weight"]
)

patient_lime_df = patient_lime_df.sort_values(
    "Absolute_LIME",
    ascending=False
)

display(patient_lime_df)

,Feature,LIME_Weight,Absolute_LIME
0,-0.70 < numeric__Glucose <= 0.01,0.155911,0.155911
1,-0.67 < numeric__BMI <= -0.01,-0.037115,0.037115
2,0.00 < categorical__Smoking Status_Never <= 1.00,-0.018179,0.018179
3,categorical__Alcohol Intake_Occasional <= 0.00,0.016616,0.016616
4,0.00 < categorical__Family History_No <= 1.00,0.015485,0.015485
5,0.00 < categorical__Hypertension_No <= 1.00,0.012617,0.012617
6,categorical__Gender_Male <= 0.00,-0.011268,0.011268
7,categorical__Family History_Yes <= 0.00,0.011185,0.011185
8,categorical__Physical Activity_Low <= 0.00,-0.010106,0.010106
9,numeric__Blood Pressure <= -0.70,0.009883,0.009883


CDSS summary

In [67]:
print("=" * 60)
print("TECHNICAL CDSS SUMMARY")
print("=" * 60)

print("Best ML Model:", best_model_name)

print(
    "Prediction:",
    "Diabetic" if prediction == 1
    else "Non-diabetic"
)

print(
    "Risk Probability:",
    f"{probability:.2%}"
)

print(
    "Risk Level:",
    risk_level
)

print("\nTop SHAP Factors:")

for _, row in patient_shap_df.head(5).iterrows():

    direction = (
        "increases"
        if row["SHAP_Value"] > 0
        else "decreases"
    )

    print(
        f"- {row['Feature']} {direction} "
        f"the predicted risk "
        f"(SHAP={row['SHAP_Value']:.4f})"
    )

print("\nTop LIME Factors:")

for _, row in patient_lime_df.head(5).iterrows():

    direction = (
        "supports"
        if row["LIME_Weight"] > 0
        else "opposes"
    )

    print(
        f"- {row['Feature']} {direction} "
        f"the prediction "
        f"(weight={row['LIME_Weight']:.4f})"
    )

TECHNICAL CDSS SUMMARY
Best ML Model: Logistic Regression
Prediction: Diabetic
Risk Probability: 96.91%
Risk Level: High Risk

Top SHAP Factors:
- numeric__Glucose decreases the predicted risk (SHAP=-4.0918)
- numeric__BMI decreases the predicted risk (SHAP=-0.6441)
- numeric__Cholesterol increases the predicted risk (SHAP=0.0279)
- numeric__Diabetes Pedigree Function increases the predicted risk (SHAP=0.0237)
- categorical__Physical Activity_Low increases the predicted risk (SHAP=0.0177)

Top LIME Factors:
- -0.70 < numeric__Glucose <= 0.01 supports the prediction (weight=0.1559)
- -0.67 < numeric__BMI <= -0.01 opposes the prediction (weight=-0.0371)
- 0.00 < categorical__Smoking Status_Never <= 1.00 opposes the prediction (weight=-0.0182)
- categorical__Alcohol Intake_Occasional <= 0.00 supports the prediction (weight=0.0166)
- 0.00 < categorical__Family History_No <= 1.00 supports the prediction (weight=0.0155)


### **Save Model, Pipeline, Results and Figures**

Create output folder

In [68]:
output_dir = "diabetes_xai_results"

os.makedirs(
    output_dir,
    exist_ok=True
)

print("Output folder created:", output_dir)

Output folder created: diabetes_xai_results


Save best model

In [69]:
joblib.dump(
    best_model,
    os.path.join(
        output_dir,
        "best_diabetes_model.pkl"
    )
)

joblib.dump(
    preprocessor,
    os.path.join(
        output_dir,
        "preprocessing_pipeline.pkl"
    )

)

print("Model and preprocessing pipeline saved.")

Model and preprocessing pipeline saved.


Save model results

In [70]:
results_df.to_csv(
    os.path.join(
        output_dir,
        "model_performance_results.csv"
    )
)

cv_results_df.to_csv(
    os.path.join(
        output_dir,
        "cross_validation_results.csv"
    )
)

shap_importance.to_csv(
    os.path.join(
        output_dir,
        "shap_feature_importance.csv"
    ),
    index=False
)

lime_results.to_csv(
    os.path.join(
        output_dir,
        "lime_explanation.csv"
    ),
    index=False
)

xai_comparison_df.to_csv(
    os.path.join(
        output_dir,
        "shap_lime_comparison.csv"
    ),
    index=False
)

print("Results saved successfully.")

Results saved successfully.


Save CDSS output

In [71]:
cdss_output = {
    "Best Model": best_model_name,
    "Prediction": (
        "Diabetic"
        if prediction == 1
        else "Non-diabetic"
    ),
    "Probability": float(probability),
    "Risk Level": risk_level
}

with open(
    os.path.join(
        output_dir,
        "cdss_prediction.json"
    ),
    "w"
) as f:

    json.dump(
        cdss_output,
        f,
        indent=4
    )

print("CDSS output saved.")

CDSS output saved.


### **Final Verification Cell**

In [72]:
print("=" * 70)
print("FINAL DISSERTATION IMPLEMENTATION CHECK")
print("=" * 70)

checks = {
    "Dataset loaded": df is not None,
    "Data preprocessing": preprocessor is not None,
    "EDA completed": True,
    "Logistic Regression": "Logistic Regression" in best_models,
    "Random Forest": "Random Forest" in best_models,
    "XGBoost": "XGBoost" in best_models,
    "5-fold CV": cv_results_df is not None,
    "Hyperparameter tuning": best_models is not None,
    "Model evaluation": results_df is not None,
    "Best model selected": best_model is not None,
    "SHAP analysis": shap_importance is not None,
    "LIME analysis": lime_results is not None,
    "Feature agreement": feature_agreement is not None,
    "SHAP consistency": shap_consistency_score is not None,
    "LIME consistency": lime_consistency_score is not None,
    "SHAP fidelity": not np.isnan(shap_fidelity),
    "LIME fidelity": not np.isnan(lime_fidelity),
    "Runtime comparison": True,
    "SHAP-LIME comparison": xai_comparison_df is not None,
    "CDSS prototype": True,
    "Model saved": os.path.exists(
        os.path.join(
            output_dir,
            "best_diabetes_model.pkl"
        )
    ),
    "Results saved": os.path.exists(
        os.path.join(
            output_dir,
            "model_performance_results.csv"
        )
    )
}

for item, status in checks.items():

    print(
        f"{'✓' if status else '✗'} {item}"
    )

print("\nCompleted dissertation coding workflow.")

FINAL DISSERTATION IMPLEMENTATION CHECK
✓ Dataset loaded
✓ Data preprocessing
✓ EDA completed
✓ Logistic Regression
✓ Random Forest
✓ XGBoost
✓ 5-fold CV
✓ Hyperparameter tuning
✓ Model evaluation
✓ Best model selected
✓ SHAP analysis
✓ LIME analysis
✓ Feature agreement
✓ SHAP consistency
✓ LIME consistency
✓ SHAP fidelity
✓ LIME fidelity
✓ Runtime comparison
✓ SHAP-LIME comparison
✓ CDSS prototype
✓ Model saved
✓ Results saved

Completed dissertation coding workflow.
